## Baseline Transformer Inference (DistilBERT)

### Model + Corpus Setup

In [21]:
# Load pre-trained DistilBERT sentiment model (fine-tuned on SST-2)
# We use this as our baseline before adding ESG-specific snippet features.

from transformers import pipeline, AutoTokenizer
import json
import pandas as pd
from pathlib import Path

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_model = pipeline("sentiment-analysis", model=model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

corpus_path = Path("../data/cleaned_v2/preprocessed_corpus.jsonl")
rows = [json.loads(line) for line in open(corpus_path, encoding="utf-8")]

print(f"✅ Corpus loaded: {len(rows)} documents")


Device set to use mps:0


✅ Corpus loaded: 9 documents


### Chunking Function

In [23]:
# Function to handle long sentences
# If tokenized sentence >512 tokens, split into 510-token chunks
# Average scores from chunks to avoid truncation bias

MAX_CHUNK_TOKENS = 510  # reserve for CLS + SEP

def get_sentiment(text, max_len=MAX_CHUNK_TOKENS):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len:
        return sentiment_model(text, truncation=True, max_length=512)[0]
    else:
        chunks = [tokens[i:i+max_len] for i in range(0, len(tokens), max_len)]
        scores = []
        for chunk in chunks:
            decoded = tokenizer.decode(chunk, skip_special_tokens=True)
            pred = sentiment_model(decoded, truncation=True, max_length=512)[0]
            pos_score = pred["score"] if pred["label"] == "POSITIVE" else 1 - pred["score"]
            scores.append(pos_score)
        avg_score = sum(scores) / len(scores)
        label = "POSITIVE" if avg_score >= 0.5 else "NEGATIVE"
        return {"label": label, "score": avg_score}

### Apply + Save Results

In [20]:
# Apply model to all sentences in corpus
# Store results (company, year, sentence, predicted label, confidence score)
# Save results as CSV in data/processed/

results = []
for doc in rows:
    company = doc["company"]
    year = doc["year"]
    for sent in doc["sentences"]:
        try:
            pred = get_sentiment(sent)
            results.append({
                "company": company,
                "year": year,
                "sentence": sent,
                "label": pred["label"],
                "score": pred["score"]
            })
        except Exception as e:
            print("⚠️ Error on sentence:", sent[:80], "|", e)

df = pd.DataFrame(results)
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "distilbert_baseline.csv"
df.to_csv(out_path, index=False)

print(f"✅ Saved baseline predictions to {out_path} | rows: {len(df)}")
print(df.head())


Token indices sequence length is longer than the specified maximum sequence length for this model (795 > 512). Running this sequence through the model will result in indexing errors


✅ Saved baseline predictions to ../data/processed/distilbert_baseline.csv | rows: 14166
  company  year                                           sentence     label  \
0  Google  2022  environmental report table of contents 1 about...  POSITIVE   
1  Google  2022      it also mentions notable targets set in 2022.  POSITIVE   
2  Google  2022  this report outlines how we're driving positiv...  POSITIVE   
3  Google  2022  for more information about our sustainability ...  POSITIVE   
4  Google  2022  for more information about our overall corpora...  POSITIVE   

      score  
0  0.902096  
1  0.972656  
2  0.999263  
3  0.615628  
4  0.914413  
